# PI3 Grupo 4 — Verificação populacional: proporção de nódulos com valor central = 3

**Objetivo:** confirmar ou refutar, com o conjunto completo de 1.010 pacientes,
a hipótese levantada no piloto de que aproximadamente 38,6% dos nódulos caem em
zona indeterminada (valor central = 3), e verificar se a amostra do piloto (os
primeiros 25 pacientes em ordem alfabética de identificador) foi representativa
ou enviesada por essa forma de seleção.

**Para colar na Seção 3.4 do Relatório de Missão**, sugestão de título:

> ### 3.4 Verificação da proporção de nódulos indeterminados no conjunto completo

---

### Por que este notebook não baixa nenhuma imagem

As notas de malignidade atribuídas por cada radiologista fazem parte da base de
anotações que vem embutida no pacote `pylidc`, independente de qualquer arquivo
DICOM estar em disco. O agrupamento de anotações em nódulos
(`cluster_annotations()`) também opera sobre as coordenadas dos contornos
salvas nessa base, não sobre a imagem em si. Isso significa que é possível
calcular o valor central de malignidade de **todos os 1.010 pacientes** sem
baixar um único exame — o que torna esta verificação rápida (minutos, não
horas) e elimina qualquer custo de armazenamento.

A ressalva correspondente: como não há imagem carregada, este notebook não
confirma nada sobre segmentação, extração de atributos ou o pipeline de
radiômica em si — apenas sobre a distribuição das notas de malignidade e o
agrupamento de anotações, que são as duas únicas coisas de que a decisão do
valor central = 3 depende.

### O que este notebook responde

1. Qual é a proporção **exata** (não estimada) de nódulos com valor central = 3
   no conjunto completo, usando os mesmos critérios de inclusão do pipeline
   (mínimo de anotadores, diâmetro mínimo, regra de agregação).
2. Se a amostra sequencial usada no piloto (pacientes 0001-0025) ficou próxima
   ou distante desse valor real — teste direto de viés de seleção.
3. Como a margem de erro de uma amostra aleatória diminuiria com o tamanho,
   para o caso de decisões futuras precisarem de estimativa rápida em vez do
   censo completo.

**Nenhum resultado deste notebook foi obtido até você executá-lo.**

## 1. Instalação e compatibilidade

Só `pylidc` é necessário. Nem `pyradiomics`, nem `SimpleITK`, nem `idc-index` —
não há imagem envolvida nesta análise.

In [ ]:
!pip install pylidc

In [ ]:
compat_src = '''
"""compat.py -- restaura APIs removidas, exigidas pelo pylidc 0.2.3."""
import configparser
import numpy as np

if not hasattr(configparser, "SafeConfigParser"):
    configparser.SafeConfigParser = configparser.ConfigParser

for _n, _t in {"float": float, "int": int, "bool": bool,
               "object": object, "str": str, "complex": complex}.items():
    if not hasattr(np, _n):
        setattr(np, _n, _t)

if not hasattr(np, "in1d"):     np.in1d = np.isin
if not hasattr(np, "alltrue"):  np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
'''

import sys, os
with open('/content/compat.py', 'w') as fh:
    fh.write(compat_src)
sys.path.insert(0, '/content')
import compat  # noqa: F401

import numpy as np
print('compat aplicado | NumPy', np.__version__)

## 2. Parâmetros — os mesmos critérios do pipeline de extração

Usar exatamente os mesmos valores do piloto garante que este número seja
comparável ao relatado na Seção 5. Se o grupo mudar esses critérios no
pipeline principal, deve mudá-los aqui também antes de comparar.

In [ ]:
MIN_ANOTADORES  = 3
DIAMETRO_MIN_MM = 3.0
REGRA_ROTULO    = 'mediana'   # mediana | media | moda -- mesma do pipeline
N_PILOTO        = 25          # tamanho da amostra original do piloto
SEMENTE         = 42          # fixa, para reprodutibilidade das amostras aleatorias

print(f'criterios: min_anotadores={MIN_ANOTADORES}, diametro_min_mm={DIAMETRO_MIN_MM}, '
      f'regra_rotulo={REGRA_ROTULO}')

## 3. Cálculo do valor central para todos os nódulos do conjunto completo

Percorre os 1.010 pacientes, agrupa as anotações de cada um em nódulos, aplica
os mesmos critérios de inclusão do pipeline e calcula o valor central de
malignidade de cada nódulo elegível. Não há leitura de imagem — apenas consulta
à base de anotações — portanto esta célula deve levar poucos minutos.

In [ ]:
import pylidc as pl
from collections import Counter
import pandas as pd
import time


def calcular_valor_central(regra):
    if regra == 'mediana':
        return lambda escores: float(np.median(escores))
    if regra == 'media':
        return lambda escores: float(np.mean(escores))
    if regra == 'moda':
        return lambda escores: float(Counter(escores).most_common(1)[0][0])
    raise ValueError(regra)


agregar = calcular_valor_central(REGRA_ROTULO)

t0 = time.time()
todos_scans = pl.query(pl.Scan).all()
print(f'{len(todos_scans)} exames na base (esperado: 1018, referentes a 1010 pacientes)')

registros = []
for i, scan in enumerate(todos_scans, 1):
    grupos = scan.cluster_annotations()
    for idx, anns in enumerate(grupos):
        if len(anns) < MIN_ANOTADORES:
            continue
        diams = [float(a.diameter) for a in anns]
        if np.mean(diams) < DIAMETRO_MIN_MM:
            continue
        escores = [int(a.malignancy) for a in anns]
        registros.append({
            'patient_id': scan.patient_id,
            'nodule_idx': idx,
            'n_anotadores': len(anns),
            'malignancy_escores': '|'.join(map(str, escores)),
            'valor_central': agregar(escores),
        })
    if i % 200 == 0:
        print(f'{i}/{len(todos_scans)} exames processados | '
              f'{len(registros)} nódulos elegíveis até agora | '
              f'{(time.time()-t0):.0f}s decorridos')

df_completo = pd.DataFrame(registros)
print(f'\nconcluído em {(time.time()-t0):.0f}s')
print(f'{df_completo.patient_id.nunique()} pacientes com ao menos 1 nódulo elegível')
print(f'{len(df_completo)} nódulos elegíveis no conjunto completo')

## 4. Proporção exata no conjunto completo

Como esta etapa processa **todos** os pacientes, e não uma amostra, o número
abaixo é o parâmetro populacional exato para os critérios definidos na Seção 2
— não uma estimativa sujeita a margem de erro.

In [ ]:
n_total = len(df_completo)
n_indeterminados = int((df_completo.valor_central == 3.0).sum())
pct_real = 100 * n_indeterminados / n_total

print('=== PROPORCAO REAL NO CONJUNTO COMPLETO (nao e estimativa) ===')
print(f'Nodulos elegiveis no conjunto completo : {n_total}')
print(f'Nodulos com valor central = 3          : {n_indeterminados}')
print(f'Proporcao real                         : {pct_real:.1f}%')
print()
print('Distribuicao completa do valor central:')
print(df_completo.valor_central.value_counts().sort_index().to_string())

## 5. A amostra do piloto foi representativa?

O piloto processou os primeiros 25 pacientes em ordem alfabética de
identificador (LIDC-IDRI-0001 a LIDC-IDRI-0025), não uma amostra aleatória.
Esta célula isola exatamente esses pacientes dentro do conjunto completo já
calculado e compara o resultado deles com o valor real da população.

In [ ]:
primeiros_pacientes = sorted(df_completo.patient_id.unique())[:N_PILOTO]
df_piloto = df_completo[df_completo.patient_id.isin(primeiros_pacientes)]

n_piloto = len(df_piloto)
k_piloto = int((df_piloto.valor_central == 3.0).sum())
pct_piloto = 100 * k_piloto / n_piloto if n_piloto else float('nan')

print('=== COMPARACAO: AMOSTRA SEQUENCIAL DO PILOTO vs. POPULACAO ===')
print(f'Amostra do piloto (primeiros {N_PILOTO} pacientes): '
      f'{k_piloto}/{n_piloto} = {pct_piloto:.1f}%')
print(f'Conjunto completo ({df_completo.patient_id.nunique()} pacientes): '
      f'{n_indeterminados}/{n_total} = {pct_real:.1f}%')
print(f'Diferenca absoluta: {abs(pct_piloto - pct_real):.1f} pontos percentuais')

## 6. Amostras aleatórias de tamanhos crescentes

Para efeito de comparação, esta célula sorteia (com semente fixa) amostras
aleatórias de diferentes tamanhos a partir do conjunto completo já calculado,
e reporta a proporção de indeterminados em cada uma, junto com o intervalo de
confiança de 95% (método de Wilson). Isso ilustra o quanto a escolha aleatória,
e não sequencial, aproxima o resultado do valor real, e o quanto a margem de
erro diminui à medida que o tamanho da amostra cresce.

In [ ]:
def wilson_ci(k, n, z=1.96):
    """Intervalo de confianca de Wilson para uma proporcao binomial.
    Mais adequado que a aproximacao normal simples quando a proporcao
    nao esta proxima de 0,5 ou quando n e pequeno."""
    if n == 0:
        return (float('nan'), float('nan'))
    phat = k / n
    denom = 1 + z**2 / n
    centro = phat + z**2 / (2 * n)
    margem = z * np.sqrt(phat * (1 - phat) / n + z**2 / (4 * n**2))
    return ((centro - margem) / denom, (centro + margem) / denom)


rng = np.random.default_rng(SEMENTE)
todos_pacientes = df_completo.patient_id.unique()

tamanhos = [25, 50, 100, 200, 500, len(todos_pacientes)]
linhas_resumo = []

for tam in tamanhos:
    tam_efetivo = min(tam, len(todos_pacientes))
    amostra_pac = rng.choice(todos_pacientes, size=tam_efetivo, replace=False)
    sub = df_completo[df_completo.patient_id.isin(amostra_pac)]
    n_sub = len(sub)
    k_sub = int((sub.valor_central == 3.0).sum())
    pct_sub = 100 * k_sub / n_sub if n_sub else float('nan')
    ic_lo, ic_hi = wilson_ci(k_sub, n_sub)
    linhas_resumo.append({
        'pacientes_sorteados': tam_efetivo,
        'nodulos_na_amostra': n_sub,
        'indeterminados': k_sub,
        'percentual': round(pct_sub, 1),
        'ic_95_inferior_%': round(100 * ic_lo, 1),
        'ic_95_superior_%': round(100 * ic_hi, 1),
    })

df_resumo = pd.DataFrame(linhas_resumo)
print('=== CONVERGENCIA DA ESTIMATIVA COM O TAMANHO DA AMOSTRA (aleatoria) ===')
print(df_resumo.to_string(index=False))
print(f'\nValor real da populacao (censo completo): {pct_real:.1f}%')

## 7. Conclusão e exportação para o relatório

Esta célula não interpreta o resultado por si — a leitura correta depende do
que as células 4, 5 e 6 mostrarem quando executadas. Ela apenas organiza os
números em formato pronto para colar na Seção 3.4 do Relatório de Missão.

In [ ]:
import json

resumo_final = {
    'criterios': {
        'min_anotadores': MIN_ANOTADORES,
        'diametro_min_mm': DIAMETRO_MIN_MM,
        'regra_rotulo': REGRA_ROTULO,
    },
    'populacao_completa': {
        'pacientes': int(df_completo.patient_id.nunique()),
        'nodulos_elegiveis': n_total,
        'indeterminados': n_indeterminados,
        'percentual_real': round(pct_real, 1),
    },
    'amostra_piloto_sequencial': {
        'pacientes': N_PILOTO,
        'nodulos': n_piloto,
        'indeterminados': k_piloto,
        'percentual': round(pct_piloto, 1),
        'diferenca_pontos_percentuais_vs_populacao': round(abs(pct_piloto - pct_real), 1),
    },
    'convergencia_amostras_aleatorias': df_resumo.to_dict(orient='records'),
}

print(json.dumps(resumo_final, indent=2, ensure_ascii=False))

# Salva no Drive, na mesma pasta do projeto, se montado
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    destino = '/content/drive/MyDrive/PI3_Grupo4/logs/verificacao_populacional_valor_central_3.json'
    with open(destino, 'w') as fh:
        json.dump(resumo_final, fh, indent=2, ensure_ascii=False)
    print(f'\nSalvo em: {destino}')
except Exception as e:
    print(f'\nNao foi possivel salvar no Drive automaticamente ({e}); '
          'copie o JSON acima manualmente se necessario.')

---
### Como ler os três resultados, ao preencher a Seção 3.4

**Se a Seção 4 mostrar percentual real próximo a 38,6%:** a hipótese do piloto
é confirmada como representativa do conjunto completo, e o número pode ser
citado no relatório final como parâmetro populacional, não apenas como achado
de amostra pequena.

**Se a Seção 5 mostrar diferença grande entre a amostra sequencial e a
população:** há evidência de viés introduzido pela seleção dos primeiros 25
pacientes em ordem alfabética — o que reforçaria a recomendação de usar
amostragem aleatória (com semente fixa) em qualquer subconjunto usado daqui
para frente, inclusive no próprio pipeline de extração radiômica.

**A Seção 6 é informativa, não decisória:** mostra como a margem de erro
diminuiria caso o grupo precise, no futuro, tomar uma decisão rápida com base
em amostra parcial em vez de esperar o processamento do conjunto inteiro.